# 05: 多分辨率 Leiden 聚类

以多个 Leiden 分辨率运行聚类并对结果进行可视化比较，帮助选择最符合生物学粒度的分群方案。

**上游**：04 产出的嵌入结果
**本 notebook 产出**：
- `obs["leiden_res_{resolution}"]` 列（各分辨率聚类标签）
- 各分辨率着色的 UMAP 图
- 聚类指标对比表与群大小分布图
- `05_clustered_v1.h5ad` checkpoint，供 06 标注使用

**扩展槽**（注释掉的 cell）：任何写出 `obs["{method}_clusters"]` 的聚类方法
均可与 `leiden_res_*` 列共存。ACDC 不是默认依赖，安装后取消注释即可插入。

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：04（多方法嵌入），读 `04_embedded_v*.h5ad`
- **下游**：06（多方法注释），产出 `05_clustered_v*.h5ad`

### 为什么要迭代回跑？
聚类分群的分辨率选择直接影响下游注释的质量。如果在 06（注释时发现
某个细胞类型被错误拆分或合并、LLM 判决分歧高）发现问题，可能需要：
- 调整 `RESOLUTIONS` 列表（加更细或更粗的分辨率）
- 换用不同的嵌入（改 `USE_REP` 指向 04 的另一个嵌入）
- 换用 04 的另一个版本（不同嵌入方法/参数组合）

### 如何回跑（三步操作）
1. **改 `UPSTREAM_PATH`**——指向要复用的上游文件版本
   （例如 `04_embedded_v2.h5ad`）
2. **改 `OUTPUT_PATH`**——bump 版本号 `_v1` → `_v2`
   （例如 `05_clustered_v2.h5ad`）
3. **调整参数**（在下方 `# === PARAMS ===` 区域改 `RESOLUTIONS` 或 `USE_REP`）
   → 重跑本 notebook（Cell → Run All）

### 版本约定
- **`_v1` / `_v2` / ...**：每次调参重跑 bump 一位版本号。
  旧版 `.h5ad` 文件**不覆盖不删除**，保留在 `results/` 目录供追溯对比。
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）。
- **`promoted`**：PI 审查后认为分群质量可接受、可传给下游使用的正式版本。
  PI 在 Jupyter 中打开 `.h5ad` 后手动改 `adata.uns["status"] = "promoted"` 再保存。
- **下游取数**：06 的 `UPSTREAM_PATH` 指向你决定采用的 05 版本即可。

### 追溯链（自动写入 h5ad 的 `adata.uns`）
本 notebook 在写出前自动记录以下字段，供后续审计查询：
- `stage` = `"05_clustered"`（本 stage 标识）
- `status` = `"experimental"`（PI 审查后改为 `"promoted"`）
- `upstream` = 本次读入的上游文件路径列表
- `version` = 与 `OUTPUT_PATH` 一致的版本号（`"v1"` / `"v2"` / ...）

如需查询"05 有哪些版本？哪些依赖 04_v1？"，
可直接在 Python 中 glob `results/` 目录检查每个 `.h5ad` 的 `adata.uns`。

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH  — 04 产出文件路径。
#                   如需回跑：指向要复用的上游版本（如 04_embedded_v2.h5ad）。
# OUTPUT_PATH    — 本 stage 产出 checkpoint 路径。
#                   版本号 _v1 与 adata.uns["version"] 保持一致。
#                   如需回跑：bump 版本号 _v1->v2，旧版不覆盖。
# USE_REP        — 用于构建邻居图的嵌入键名（obsm 中的 key）。
#                   如需换用不同嵌入：改为 04 产出的其他嵌入，
#                   如 "X_pca_harmony"、"X_scVI"、"X_pca"。
# RESOLUTIONS    — 要尝试的 Leiden 分辨率列表。
#                   分辨率越低，分群越粗（如 0.2-0.4 捕获大谱系）；
#                   分辨率越高，分群越细（如 1.0+ 区分亚型）。
#                   PI 查看各分辨率 UMAP 后选择最合理的值。
# RANDOM_SEED    — 固定随机种子，保证可复现。

UPSTREAM_PATH = "results/04_embedded_v1.h5ad"
OUTPUT_PATH   = "results/05_clustered_v1.h5ad"

USE_REP      = "X_pca_harmony"      # Harmony 批次校正嵌入（scVI 在 04 已禁用）
RESOLUTIONS  = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.5, 2.0]
RANDOM_SEED  = 42

In [ ]:
# 确保框架 src/ 在 sys.path 上，CWD 为项目根目录。
# 自动检测两种运行场景：从 notebooks/（Jupyter）还是项目根目录（nbconvert）启动。
import sys, os
_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/figures/sweep_05", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

In [ ]:
# 导入（scanpy 原生 API + 框架函数仅在真正有缺口时使用）。
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
import warnings

# 直接从 scorers 模块导入——无回调，在 for 循环中直接调用
from scrna_integration.scorers import clustering_metrics

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

## 加载上游数据

读取 04 产出的嵌入结果，确认数据维度与嵌入键名。
后续所有操作（邻居图、聚类、UMAP）均基于此 AnnData 对象。

In [ ]:
print("加载上游:", UPSTREAM_PATH)
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")
print(f"obsm keys: {list(adata.obsm.keys())}")
print(f"use_rep='{USE_REP}' -- 存在: {USE_REP in adata.obsm}")

## 计算邻居图

在 04 选定嵌入（`USE_REP`）上构建 k 近邻图。
**为什么所有分辨率共享同一个邻居图？**
不同分辨率只改变聚类的粒度（通过 Leiden 算法的 resolution 参数），
不需要每次重新计算细胞间距离关系。共享避免了重复计算和内存浪费。

`n_pcs=None`：嵌入已是低维表示，无需再做 PCA 降维。

In [ ]:
# 在选定的嵌入上计算邻居图，所有分辨率共享。
print(f"\n===== Neighbors on {USE_REP} =====")
sc.pp.neighbors(
    adata,
    use_rep=USE_REP,
    n_pcs=None,
    random_state=RANDOM_SEED,
)
print(f"Neighbor graph: {adata.obsp['connectivities'].shape}")
print(f"  n_neighbors={adata.uns['neighbors']['params']['n_neighbors']}")

## 多分辨率 Leiden 聚类

每个分辨率各做一次 Leiden 聚类。所有 `leiden_res_*` 列共存以供对比。
使用 `flavor="igraph"` 以兼容 scanpy >= 1.10。

**为什么做多个分辨率？** 没有"正确"的分辨率——不同生物学问题需要
不同粒度。粗分辨率（0.2-0.4）捕获大细胞谱系；细分辨率（1.0+）
区分亚型。PI 查看 UMAP 后根据生物学背景选择最合理的一个或几个。

In [ ]:
# 多分辨率 Leiden 聚类：每个分辨率一个 obs 列。
print(f"\n===== Leiden: {len(RESOLUTIONS)} resolutions =====")

for res in RESOLUTIONS:
    key = f"leiden_res_{res}"
    print(f"  resolution={res} -> obs['{key}']")
    sc.tl.leiden(
        adata,
        resolution=res,
        key_added=key,
        flavor="igraph",
        random_state=RANDOM_SEED,
    )
    n_clusters = adata.obs[key].nunique()
    print(f"    clusters: {n_clusters}")

leiden_columns = [c for c in adata.obs.columns if c.startswith("leiden_res_")]
print(f"\nLeiden columns produced: {leiden_columns}")

## 聚类指标对比——辅助选择最佳分辨率

对每个分辨率计算聚类质量指标，与可视化结果互为补充。

**直接遍历分辨率列表**——没有回调、没有封装。学生逐行可读。

**指标说明**（数据允许时计算）：
- **silhouette score**（轮廓系数）：衡量簇的紧密程度与分离度。值越接近 1 越好。
  在 PCA 空间上计算，反映嵌入空间中簇的结构质量。
- **ARI**（调整兰德指数）：与已知标签的一致性。需要 `obs` 中存在参考标签列。

**如何用指标辅助决策**：
- silhouette 随分辨率升高通常会缓慢下降——这是正常的（更细的簇边界更模糊）。
  重点看下降趋势中是否存在"拐点"（分辨率增加但 silhouette 骤降），而非绝对值。
- ARI 仅在存在参考标签时有意义。
- **最终决策以 UMAP 可视化为主，指标为参考**——生物学的分群合理性不能仅由数值指标决定。

对比报告写入 `results/figures/sweep_05/sweep_report.md`。

In [ ]:
# 显式 for 循环：遍历各分辨率，运行 Leiden 并计算聚类指标。
# 每个分辨率在独立拷贝上运行，避免列名冲突。
print("\n===== 显式遍历分辨率 + clustering_metrics =====\n")

results = []
for res in RESOLUTIONS:
    print(f"--- resolution={res} ---")

    # 拷贝 AnnData，在该分辨率运行 Leiden
    adata_copy = adata.copy()
    key = f"leiden_res_{res}"
    sc.tl.leiden(
        adata_copy,
        resolution=res,
        key_added=key,
        flavor="igraph",
        random_state=RANDOM_SEED,
    )

    # 直接调用聚类指标函数——显式传入 cluster_key=key 避免 auto-detect 误选其他 leiden 列
    m = clustering_metrics(adata_copy, cluster_key=key)
    results.append({"resolution": res, **m})

    n_cl = adata_copy.obs[key].nunique()
    metrics_str = ", ".join(f"{k}={v:.4f}" for k, v in m.items()
                            if isinstance(v, float) and not np.isnan(v))
    print(f"  簇数: {n_cl}  指标: {metrics_str}")

# 收集为 DataFrame 对比表
sweep_df = pd.DataFrame(results)
os.makedirs("results/figures/sweep_05", exist_ok=True)

# 写 Markdown 报告
lines = ["# 05 聚类对比报告\n",
         f"**{len(RESOLUTIONS)} 个分辨率** 已评估。\n",
         "## 指标表\n"]
lines.append("| " + " | ".join(sweep_df.columns) + " |")
lines.append("|" + "|".join(" --- " for _ in sweep_df.columns) + "|")
for _, row in sweep_df.iterrows():
    vals = []
    for col in sweep_df.columns:
        v = row[col]
        if isinstance(v, float):
            vals.append(f"{v:.4f}" if not np.isnan(v) else "N/A")
        else:
            vals.append(str(v))
    lines.append("| " + " | ".join(vals) + " |")
with open("results/figures/sweep_05/sweep_report.md", "w") as f:
    f.write("\n".join(lines) + "\n")

# 绘制指标随分辨率变化的折线图——辅助 PI 识别拐点
metric_cols = [c for c in sweep_df.columns
               if c != "resolution" and not sweep_df[c].isna().all()]
if metric_cols:
    fig, axes = plt.subplots(1, len(metric_cols), figsize=(5 * len(metric_cols), 4))
    if len(metric_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, metric_cols):
        ax.plot(sweep_df["resolution"], sweep_df[col], "o-", color="#2c7bb6", markersize=6)
        ax.set_xlabel("resolution")
        ax.set_ylabel(col)
        ax.set_title(f"{col} vs resolution")
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig_path = "results/figures/sweep_05/metrics_vs_resolution.png"
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"\n指标-分辨率折线图: {fig_path}")
    plt.close("all")

# 展示对比表
print("\n聚类指标对比表:")
try:
    from IPython.display import display as ipy_display
    ipy_display(sweep_df)
except ImportError:
    print(sweep_df)

print("\n对比报告: results/figures/sweep_05/sweep_report.md")
adata.uns["05_sweep_v1"] = {
    "resolutions_swept": RESOLUTIONS,
    "scorer": "clustering_metrics",
    "report_dir": "results/figures/sweep_05",
    "timestamp": datetime.datetime.now().isoformat(),
}

## 扩展槽——其他聚类方法

任何写出 `obs["{method}_clusters"]` 的聚类方法均可与 `leiden_res_*` 列共存
并以同样方式参与对比。

**ACDC**（已注释）：全局搜索最优分区。**不**是默认依赖（在 GCPL 数据上
运行太慢）。PI 安装后取消注释，新列自动进入 06。

**新方法的通用模式**：写 `obs["{method}_clusters"]` = labels，
然后加入显式 for 循环的 candidates 列表或单独对比——
与 04 嵌入同样的并行槽位约定，零框架改动。

In [ ]:
# # === ACDC clustering (commented out -- NOT a default dependency) ===
# # PREREQUISITE: pip install acdc_py
# # Enable by removing comments below.
#
# # import acdc_py  # 包名/导入名以 PyPI 实际为准，启用前先确认；ACDC 非默认依赖
# # # ACDC searches for an optimal partition.
# # acdc_result = ACDC.ACDC(adata, ...)
# # adata.obs["acdc_clusters"] = acdc_result.labels
# # print(f"ACDC: {adata.obs['acdc_clusters'].nunique()} clusters")
#
# print("ACDC cell is commented out. "
#       "Uncomment when acdc_py is installed and suitable for this dataset.")
#
# # === Add any future method here ===
# # Pattern: write adata.obs["{method}_clusters"] = labels
# # Then add to sweep candidates or compare standalone.
# # Example: adata.obs["foocluster_clusters"] = foo_cluster.fit_predict(
# #     adata.obsm[USE_REP])

## 各分辨率 UMAP 可视化

**按分辨率逐个着色 UMAP**——每个分辨率算完即出图。
PI 可以跑一个分辨率看一个，判断该分辨率的聚类是否合理，再决定最终采纳哪个。

**怎么看 UMAP 判断分群质量**：
- 同一簇在 UMAP 上应该聚在一起（簇内紧凑）
- 不同簇之间应该有清晰的边界或过渡
- 如果某个分辨率把一个明显的谱系拆成多块 → 过分（分辨率太高）
- 如果某个分辨率把明显不同的细胞群合并 → 欠分（分辨率太低）
- 在 UMAP 上看簇的分布，结合下方的群大小分布图，综合判断

In [ ]:
# 从已有邻居图计算 UMAP（所有分辨率共享同一嵌入）。
# 然后逐个分辨率着色，每个分辨率即时出图，方便 PI 对比选择。
print("\n===== UMAP per resolution =====")

sc.tl.umap(adata, random_state=RANDOM_SEED)

leiden_cols = sorted(
    [c for c in adata.obs.columns if c.startswith("leiden_res_")],
    key=lambda x: float(x.split("_")[-1]),
)

for col in leiden_cols:
    n_clusters = adata.obs[col].nunique()
    res_str = col.split("_")[-1]
    print(f"\n--- resolution={res_str}: {n_clusters} clusters ---")

    sc.pl.umap(
        adata, color=col,
        title=f"Leiden res={res_str}（{n_clusters} 群）",
        legend_loc="on data" if n_clusters <= 10 else "right margin",
        frameon=False,
        save=f"_05_{col}.png",
    )
    # 从 scanpy 默认 figures/ 目录移到 results/figures/
    src = f"figures/umap_05_{col}.png"
    dst = f"results/figures/05_umap_{col}.png"
    if os.path.exists(src):
        os.rename(src, dst)
        print(f"    图已保存: {dst}")
    else:
        print(f"    注意: 图未在默认位置 {src} 找到")
    plt.close("all")

print(f"\nUMAP 出图完成，共 {len(leiden_cols)} 个分辨率。")

## 群大小分布——辅助判断分辨率合理性

除了 UMAP 可视化，各分辨率下的群大小分布也是选择分群方案的重要参考：

- **如果存在极大群和极小群并存**（一个群占了 50% 以上细胞，另一群只有几十个）→ 可能分辨率不合适
- **如果群大小相对均匀** → 分群粒度与数据结构匹配较好
- **极小的群**（< 1% 总细胞数）可能是噪声或稀有细胞亚型——需要结合生物学背景判断

下方的条形图展示每个分辨率下各群的细胞数分布。

In [ ]:
# 各分辨率下的群大小分布——帮助 PI 判断分群是否合理。
print("\n===== 群大小分布 =====")

n_res = len(RESOLUTIONS)
n_cols = 3
n_rows = (n_res + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
if n_rows == 1 and n_cols == 1:
    axes = [axes]
else:
    axes = axes.flatten()

for i, res in enumerate(RESOLUTIONS):
    key = f"leiden_res_{res}"
    cluster_sizes = adata.obs[key].value_counts().sort_index()
    n_clusters = len(cluster_sizes)

    ax = axes[i]
    colors = plt.cm.tab20(np.linspace(0, 1, n_clusters))
    ax.bar(range(n_clusters), cluster_sizes.values, color=colors)
    ax.set_title(f"resolution={res}（{n_clusters} 群）")
    ax.set_xlabel("群编号")
    ax.set_ylabel("细胞数")
    ax.set_xticks(range(n_clusters))
    ax.tick_params(axis="x", labelsize=8)

# 隐藏多余的子图
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
fig_path = "results/figures/05_cluster_sizes.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print(f"群大小分布图保存至: {fig_path}")
plt.close("all")

## 运行元数据

版控键记录聚类参数与追溯链，供后续审计查询。

In [ ]:
# 记录聚类运行元数据——包含参数、版本与追溯链。
print("\n===== Run metadata =====")

adata.uns["leiden_v1"] = {
    "use_rep": USE_REP,
    "resolutions": RESOLUTIONS,
    "flavor": "igraph",
    "n_neighbors": adata.uns["neighbors"]["params"]["n_neighbors"],
    "timestamp": datetime.datetime.now().isoformat(),
}

# 统一追踪字段——stage + version（与上游字段合并，保持 pipeline 编号命名一致）
adata.uns["stage"] = "05_clustered"     # 本 stage 标识
adata.uns["version"] = "v1"             # 与 OUTPUT_PATH 版本号一致
adata.uns["upstream"] = [UPSTREAM_PATH]
adata.uns["status"] = "experimental"    # PI 审查后改为 "promoted"

# 各分辨率的簇数汇总
cluster_summary = {}
for col in leiden_cols:
    cluster_summary[col] = int(adata.obs[col].nunique())
adata.uns["leiden_v1"]["cluster_counts"] = cluster_summary

print("Clusters per resolution:")
for col, n in cluster_summary.items():
    print(f"  {col}: {n}")
print(f"status: {adata.uns['status']}")

In [ ]:
# 写入前自检——确保 X 保持稀疏 float32，防止意外 densify 导致内存暴涨。
import scipy.sparse as sp
import numpy as np
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 意外退化: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("自检通过: X 为稀疏 CSR float32。")

In [ ]:
# 写出 checkpoint——lzf 压缩兼顾速度与空间，保留稀疏 CSR 布局。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写入: {OUTPUT_PATH}")

import os
assert os.path.exists(OUTPUT_PATH), f"文件未找到: {OUTPUT_PATH}"
print(f"验证通过: {OUTPUT_PATH}（{os.path.getsize(OUTPUT_PATH):,} bytes）")

In [ ]:
# 跨 stage 边界释放内存。
# 如果 Jupyter 内核会话中接着跑下一 stage（06），
# 这一步避免两个 stage 的 AnnData 同时驻留内存导致 OOM。
del adata
import gc
gc.collect()
print("内存已释放。")